# Week 2b.2 — Agent Basics in LangChain

Now that we've covered the API basics in LangChain, we're ready to implement our first agent.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("TAVILY_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [4]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=True)

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


## 1. The problem

LLMs can serve as the brain but their knowledge and reach are limited.

In [5]:
from pprint import pprint

response = model.invoke("What time is it right now?")
pprint(response.content)

("I don't have access to real-time data like the current date or time. To get "
 "the exact time where you are, I'd recommend:  \n"
 '- Checking a trusted clock app or website (e.g., weather.com, '
 'worldclock.org)  \n'
 '- Asking someone locally for help  \n'
 "- Using your device's built-in clock functionality  \n"
 '\n'
 "If you're near a specific timezone (e.g., New York, Tokyo), let me know and "
 'I can suggest approximate times based on common references! 😊')


The model has no clock. Its weights were frozen months ago and nothing in the context tells it the time. It can only decline or guess.

We can surpass this limitations by augmenting the LLM with a tool.

## 2. Defining a tool

A tool starts life as an ordinary Python function.

In [6]:
from datetime import datetime

def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [7]:
get_current_time()

'Tuesday, September 22, 2026 at 04:04 PM'

### Requirements for Tool Definition
To make this usable by a model, LangChain needs three things: 
- a name (start with a verb and describe the core of the task)
- a description in the docstring
- an argument schema. 

The `@tool` decorator builds all three from your function. 

**The docstring is not a comment here; the model reads it to decide when to call the tool. Write it carefully.**

In [8]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

#TODO: define `get_current_time` as a tool

@tool
def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

Inspect what the decorator built:

In [9]:
print(get_current_time.name)
print(get_current_time.description)
print(get_current_time.args)

get_current_time
Return the current local date and time.
{}


In [10]:
get_current_time.invoke({})

'Tuesday, September 22, 2026 at 04:04 PM'

## 3. Give the tool to the model

`bind_tools` attaches tool schemas to the model as a **list** of tools.

The model does not gain the ability to run anything. It only gains the ability to ask.

In [11]:
#TODO: bind the tool to the model
model_with_tools = model.bind_tools([get_current_time])

In [12]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
response = model_with_tools.invoke([question])

In [13]:
pprint(response)

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the current time. I have access to a function called "get_current_time" that can return the current local date and time. This would be perfect for answering this question. Let me call this function to get the current time.'}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:04:52.597752Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3455559708, 'load_duration': 1812708, 'prompt_eval_count': 263, 'prompt_eval_duration': 910480000, 'eval_count': 69, 'eval_duration': 2514467000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cab8-9375-7950-be6e-3c457eb55710-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'bb59b137-6b62-4fe5-bd21-454d4170de8f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 263, 'output_tokens': 69, 'total_tokens': 332})


In [14]:
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_current_time (bb59b137-6b62-4fe5-bd21-454d4170de8f)
 Call ID: bb59b137-6b62-4fe5-bd21-454d4170de8f
  Args:


Note: `content` is empty. Instead of answering, the model produced a `tool_calls` entry naming our tool.

Remember the model proposes; it never executes.

In [15]:
pprint(response.response_metadata)

{'created_at': '2026-09-22T20:04:52.597752Z',
 'done': True,
 'done_reason': 'stop',
 'eval_count': 69,
 'eval_duration': 2514467000,
 'load_duration': 1812708,
 'logprobs': None,
 'model': 'qwen3.5:4b',
 'model_name': 'qwen3.5:4b',
 'model_provider': 'ollama',
 'prompt_eval_count': 263,
 'prompt_eval_duration': 910480000,
 'total_duration': 3455559708}


In [16]:
print(response.tool_calls)

[{'name': 'get_current_time', 'args': {}, 'id': 'bb59b137-6b62-4fe5-bd21-454d4170de8f', 'type': 'tool_call'}]



Executing is our job. Run the tool with the call the model requested:

In [17]:
call = response.tool_calls[0]
tool_result = get_current_time.invoke(call)
print(tool_result)

content='Tuesday, September 22, 2026 at 04:04 PM' name='get_current_time' tool_call_id='bb59b137-6b62-4fe5-bd21-454d4170de8f'


That produced a `ToolMessage`. Now we can combine the full conversation, including the model's tool call and the tool's result, and send it back:

In [18]:
messages = [question, response, tool_result]

final = model_with_tools.invoke(messages)


In [19]:
model.invoke(messages)


AIMessage(content="It's Tuesday, September 22, 2026 at 04:04 PM!", additional_kwargs={'reasoning_content': 'The function returned the current date and time. I can now provide this information to the user in a clear and helpful way.'}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:04:57.91184Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2417670125, 'load_duration': 1420125, 'prompt_eval_count': 121, 'prompt_eval_duration': 460431000, 'eval_count': 52, 'eval_duration': 1920440000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cab8-ac45-7d40-a10e-5836c6f6983f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 52, 'total_tokens': 173})

In [20]:
pprint(final)

AIMessage(content='It is currently Tuesday, September 22, 2026 at 04:04 PM.', additional_kwargs={'reasoning_content': 'The function returned the current local time: Tuesday, September 22, 2026 at 04:04 PM. I can now share this information with the user in a clear and helpful way.'}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:04:55.488679Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2862588916, 'load_duration': 1209333, 'prompt_eval_count': 367, 'prompt_eval_duration': 218088000, 'eval_count': 73, 'eval_duration': 2637482000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cab8-a111-7c72-ae5f-e1c824e4e44f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 367, 'output_tokens': 73, 'total_tokens': 440})


In [21]:
print(final.content)

It is currently Tuesday, September 22, 2026 at 04:04 PM.


### Interim Summary

1. The model saw the question and proposed a tool call.
2. Our code executed the tool.
3. The result went back into the conversation.
4. The model read the result and answered.

Model proposes, runtime executes, result returns. That loop is an agent.

## 4. `create_agent`: Defining the Agentic Loop

The loop so far looked like this:
 propose, execute, return, repeat until the model answers. 
 
 Because this is such a common pattern, LangChain provides a dedicated method to run the loop: `create_agent`
 
 - https://reference.langchain.com/python/langchain/agents/factory/create_agent

 Basic signature: `create_model(model, tools, system_prompt)`


In [22]:
system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""

In [23]:
from langchain.agents import create_agent

#TODO: define an agent with model, tool list, and system prompt

agent = create_agent(model=model,
                        tools= [get_current_time],
                        system_prompt=system_prompt)


In [24]:
result = agent.invoke({"messages": question})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What time is it right now?
================================== Ai Message ==================================
Tool Calls:
  get_current_time (d72ad383-2315-40cd-a8df-a6dc32b7619a)
 Call ID: d72ad383-2315-40cd-a8df-a6dc32b7619a
  Args:
================================= Tool Message =================================
Name: get_current_time

Tuesday, September 22, 2026 at 04:05 PM
================================== Ai Message ==================================

It's currently Tuesday, September 22, 2026 at 4:05 PM.


## 6. Multiple tools

Real agents have several tools and choose between them. Models are bad at date arithmetic, so we'll give our agent a second tool for that.

In [25]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

#TODO: define `get_current_time` as a tool

@tool
def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [26]:
@tool
def count_days_until_given_date(date: str) -> str:
    """Return the number of days from today until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(date, "%Y-%m-%d").date()
    delta = (target - datetime.now().date()).days
    return f"{delta} days"


In [27]:
@tool
def count_days_until_given_date(current_date: str, target_date: str) -> str:
    """Return the number of days from current_date provided as an argument until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(target_date, "%Y-%m-%d").date()
    current = datetime.strptime(current_date, "%Y-%m-%d").date()

    delta = (target - current).days
    return f"{delta} days"


In [28]:
agent = create_agent(model=model, 
                    tools=[get_current_time,count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )


In [29]:
question_day = HumanMessage(content="How many days until the midterm on 2026-10-30?")
response = agent.invoke({"messages":question_day })


In [30]:
pprint(response)

{'messages': [HumanMessage(content='How many days until the midterm on 2026-10-30?', additional_kwargs={}, response_metadata={}, id='d3aec53d-cf0b-42e2-9ee4-4710ebae881f'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to know how many days until the midterm on 2026-10-30. I need to calculate the number of days from the current date until that target date.\n\nFirst, I need to get the current date and time using the get_current_time function, then I can use count_days_until_given_date to calculate the days until 2026-10-30.'}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:05:08.484677Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5176227042, 'load_duration': 1845542, 'prompt_eval_count': 389, 'prompt_eval_duration': 1301215000, 'eval_count': 104, 'eval_duration': 3833779000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cab8-cacb-70e1-a090-e434d76d69c8-0', tool

In [31]:
pprint(response["messages"][3].additional_kwargs)

{'reasoning_content': 'Now I have the current date: September 22, 2026. I need '
                      'to calculate the number of days until October 30, 2026.'}


In [32]:
pprint(response)

{'messages': [HumanMessage(content='How many days until the midterm on 2026-10-30?', additional_kwargs={}, response_metadata={}, id='d3aec53d-cf0b-42e2-9ee4-4710ebae881f'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to know how many days until the midterm on 2026-10-30. I need to calculate the number of days from the current date until that target date.\n\nFirst, I need to get the current date and time using the get_current_time function, then I can use count_days_until_given_date to calculate the days until 2026-10-30.'}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:05:08.484677Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5176227042, 'load_duration': 1845542, 'prompt_eval_count': 389, 'prompt_eval_duration': 1301215000, 'eval_count': 104, 'eval_duration': 3833779000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0cab8-cacb-70e1-a090-e434d76d69c8-0', tool

The model picked the right tool and filled the argument from the question. Since executing calls is always the same steps, write a small helper:

## 6. Tools that call real APIs

**Motivation for web search:**
GPT-4.1-mini has a knowledge cutoff date of June 1, 2024.
 - https://developers.openai.com/api/docs/models/gpt-4.1-mini

 Qwen3.5's knowledge cutoff date is late 2024 to early 2025.

 Gemini 3.1 Flash-Lite has a knowledge cutoff date of January 2025.

They won't be able to answer questions about events that may have happened after the cutoff date

In [33]:
# same agent as before
agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )

In [34]:
questions = [HumanMessage("Which country won the FIFA World Cup 2026"), 
            HumanMessage("Who won the Super Bowl 2026?"), 
            HumanMessage("Who is the current president of the US")]

In [35]:
result = agent.invoke({"messages": questions[-1]})
pprint(result)


{'messages': [HumanMessage(content='Who is the current president of the US', additional_kwargs={}, response_metadata={}, id='535456de-721d-47d1-9583-483ce5e1a779'),
              AIMessage(content='Joe Biden is the current president of the United States. He took office on January 20, 2021, and has served his first term under the Biden administration.', additional_kwargs={'reasoning_content': "The user is asking about who is currently the president of the United States. This is general knowledge that I can answer directly without needing to use any tools. As of my knowledge, Joe Biden is the 46th President of the United States. However, I should be careful here - I should provide accurate and up-to-date information.\n\nThe US presidential term was Joe Biden from January 2021 through January 2025. Let me provide a simple, accurate answer without needing to use any tools since this is general knowledge that doesn't require date calculations or current time queries specifically."}, respons

In [36]:
pprint(result["messages"][-1].content)

('Joe Biden is the current president of the United States. He took office on '
 'January 20, 2021, and has served his first term under the Biden '
 'administration.')


In [37]:
# loop through questions and then invoke the agent
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    print("")

'Which country won the FIFA World Cup 2026'
('The FIFA World Cup 2026 has not happened yet. The tournament is scheduled to '
 'take place in June and July 2026, and therefore there is no official winner '
 'at this time.')

'Who won the Super Bowl 2026?'
('Super Bowl 2026 has not happened yet. Since today is August 9, 2025, the '
 'Super Bowl 2026 will take place in February 2026, and its results are still '
 'TBD (to be determined).')

'Who is the current president of the US'
('The current President of the United States is Joe Biden. He was inaugurated '
 'as the 46th U.S. President on January 20, 2021, after winning the election '
 'against former President Donald Trump in November 2020.')



### Tavily Web Search API

Let's add web search capability to our agent. We'll use Tavily as our web search API.
- https://app.tavily.com/

We'll need to provide an API key in the `.env` file, which has a `TAVILY_API_KEY` field.

#### TavilySearch
Because Tavily is specialized for agentic web search, LangChain supports it.

The `TavilySearch` method queries the Tavily Search API and gets back json.

https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch


In [38]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results


In [39]:
data = search_the_web.invoke({"query":"Who won the super bowl 2026"})

In [40]:
data

{'query': 'Who won the super bowl 2026',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.cincinnati.com/story/sports/nfl/2026/09/02/who-won-super-bowl-2026/91578105007',
   'title': 'Who won the Super Bowl in 2026? Final score, MVP ...',
   'content': "After a long, lonely offseason, you may have forgotten where the NFL drama left off after the last big game. While all eyes are focused on making it to Super Bowl 61 – which is still months away – here's a little refresher to get back up to speed.\n\n## Who won Super Bowl 2026?\n\nThe Seattle Seahawks defeated the New England Patriots, 29-23, in Super Bowl 60, which took place at Levi's Stadium in Santa Clara, California, on Feb. 8, 2026. [...] Seattle Seahawks defeated the New England Patriots, 29-23, in Super Bowl 60.\n\n## Who won Super Bowl 60 MVP?\n\nSeattle Seahawks running back Kenneth Walker III was named the MVP of Super Bowl 60.\n\n## Who won Super Bowl 2025?\n\nThe Philadelphia E

In [41]:
search_docs = data.get("results", data)


In [42]:
pprint(search_docs)

[{'content': 'After a long, lonely offseason, you may have forgotten where the '
             'NFL drama left off after the last big game. While all eyes are '
             'focused on making it to Super Bowl 61 – which is still months '
             "away – here's a little refresher to get back up to speed.\n"
             '\n'
             '## Who won Super Bowl 2026?\n'
             '\n'
             'The Seattle Seahawks defeated the New England Patriots, 29-23, '
             "in Super Bowl 60, which took place at Levi's Stadium in Santa "
             'Clara, California, on Feb. 8, 2026. [...] Seattle Seahawks '
             'defeated the New England Patriots, 29-23, in Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 60 MVP?\n'
             '\n'
             'Seattle Seahawks running back Kenneth Walker III was named the '
             'MVP of Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 2025?\n'
             '\n'
             'T

Our agent has three tools now:

In [44]:
from langchain.agents import create_agent

agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date, search_the_web],
                    system_prompt = "You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary."
)

In [45]:
result = agent.invoke({"messages": questions[0]})

In [46]:
pprint(result)

{'messages': [HumanMessage(content='Which country won the FIFA World Cup 2026', additional_kwargs={}, response_metadata={}, id='8e2679b4-6672-4af3-b2c9-1252aca30cc9'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "The user is asking about which country won the FIFA World Cup 2026. I need to think about this:\n\n1. The FIFA World Cup 2026 hasn't happened yet - it's scheduled for June/July 2026\n2. Since we're currently in 2025 (based on my current date knowledge), the 2026 World Cup hasn't been played yet\n3. This means I can search the web to get accurate information about whether there's already a winner announced or if this is just scheduled\n\nLet me search for this information to provide accurate and up-to-date information to the user."}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-22T20:07:13.442614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7758901000, 'load_duration': 1494834, 'prompt_eval_count': 436, 'prompt_ev

In [47]:
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    

'Which country won the FIFA World Cup 2026'
('**Spain won the FIFA World Cup 2026!**\n'
 '\n'
 'In a dramatic final held on July 19, 2026 at the MetLife Stadium in New '
 'Jersey, Spain defeated Argentina 1-0 after extra time thanks to a goal by '
 'Ferran Torres. \n'
 '\n'
 "This victory marked Spain's second World Cup title overall (having "
 'previously won in 2010). The 2026 tournament was special as it was contested '
 'by 48 teams across three North American nations (USA, Mexico, and Canada).')
'Who won the Super Bowl 2026?'
('**The Seattle Seahawks won Super Bowl 2026!** 🏈🏆\n'
 '\n'
 'They defeated the New England Patriots with a final score of **29-13** on '
 "February 8, 2026, at Levi's Stadium in Santa Clara, California. The game was "
 'described as a franchise-defining moment featuring a "Dark Side" defense '
 'that controlled the night.')
'Who is the current president of the US'
('Based on the information I found, **Donald Trump** is the current President '
 'of the United

In [ ]:
# the response from the model
pprint(result["messages"][-1].content)

## 7. ICA: Build a weather tool

Your turn. [wttr.in](https://wttr.in) is a weather service that takes a city name directly and needs no key. One request:

```
https://wttr.in/Boston?format=j1
```

returns JSON. 

The current conditions live in `data["current_condition"][0]`, with fields including `temp_F`, `weatherDesc`, and `windspeedMiles`.

Boston or Boston,MA are acceptable. If city name matches multiple locations, it'll likely default to the one closest to your IP address.

### Your task:
Write a tool `get_weather(city)` that returns the current conditions for that city. Steps:

1. Build the URL from the city argument and fetch it with `requests.get(...).json()`.
2. Pull out temperature, conditions, and wind, and return a readable string.
3. Write the docstring so the model knows when to reach for this tool.
4. Add it to the agent and ask a question that needs it.

Skeleton:

In [50]:
import requests
data = requests.get("https://wttr.in/Boston?format=j1").json()

In [51]:
data

{'current_condition': [{'FeelsLikeC': '10',
   'FeelsLikeF': '50',
   'cloudcover': '98',
   'humidity': '59',
   'observation_time': '07:12 PM',
   'precipInches': '0.0',
   'precipMM': '0.0',
   'pressure': '1028',
   'pressureInches': '30',
   'temp_C': '15',
   'temp_F': '60',
   'uvIndex': '3',
   'visibility': '10',
   'visibilityMiles': '6',
   'weatherCode': '122',
   'weatherDesc': [{'value': 'Overcast '}],
   'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}],
   'winddir16Point': 'NE',
   'winddirDegree': '52',
   'windspeedKmph': '23',
   'windspeedMiles': '14'}],
 'nearest_area': [{'areaName': [{'value': 'Boston'}],
   'country': [{'value': 'United States of America'}],
   'latitude': '42.358',
   'longitude': '-71.060',
   'population': '571281',
   'region': [{'value': 'Massachusetts'}],
   'weatherUrl': [{'value': 'https://www.worldweatheronline.com/v2/weather.aspx?q=42.358,-71.06'}]}],
 'request

In [52]:
pprint(data["current_condition"][0])

{'FeelsLikeC': '10',
 'FeelsLikeF': '50',
 'cloudcover': '98',
 'humidity': '59',
 'observation_time': '07:12 PM',
 'precipInches': '0.0',
 'precipMM': '0.0',
 'pressure': '1028',
 'pressureInches': '30',
 'temp_C': '15',
 'temp_F': '60',
 'uvIndex': '3',
 'visibility': '10',
 'visibilityMiles': '6',
 'weatherCode': '122',
 'weatherDesc': [{'value': 'Overcast '}],
 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}],
 'winddir16Point': 'NE',
 'winddirDegree': '52',
 'windspeedKmph': '23',
 'windspeedMiles': '14'}


In [58]:
import requests


@tool
def get_weather(city: str) -> str:
    """Get the weather information for the given city and return it as a string"""
    url = f"https://wttr.in/{city}?format=j1"
    data = requests.get(url).json()
    result = data["current_condition"][0]
    return result


In [57]:
# Test your tool directly first:
print(get_weather.invoke({"city": "Boston"}))

# Then give it to the agent:
agent = create_agent(model=model, tools=[get_weather])
result = agent.invoke({"messages": [HumanMessage(content="Should I bring a jacket in Boston tonight?")]})
print(result["messages"][-1].content)

{'FeelsLikeC': '10', 'FeelsLikeF': '50', 'cloudcover': '98', 'humidity': '59', 'observation_time': '07:12 PM', 'precipInches': '0.0', 'precipMM': '0.0', 'pressure': '1028', 'pressureInches': '30', 'temp_C': '15', 'temp_F': '60', 'uvIndex': '3', 'visibility': '10', 'visibilityMiles': '6', 'weatherCode': '122', 'weatherDesc': [{'value': 'Overcast '}], 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}], 'winddir16Point': 'NE', 'winddirDegree': '52', 'windspeedKmph': '23', 'windspeedMiles': '14'}
Based on the weather forecast for Boston tonight, I'd recommend bringing a light jacket. Here's what to expect:

- **Temperature**: 60°F (15°C) with a **feels like** temperature of 50°F due to wind chill
- **Conditions**: Overcast with no precipitation expected
- **Wind**: 14 mph from the northeast

While 60°F isn't particularly cold, the combination of overcast skies and windy conditions will make it feel significantly coo